# 07 — Advanced: MOT Algorithms & Deep Fusion

> **참고:** fmcw-radar-sensor (Phase 12–29), PointPainting 포트폴리오
> **언어:** Python
> **목표:** 확률론적 MOT (JPDA, PHD, PMBM) + 딥러닝 기반 융합

---

## 7-1. 왜 고급 MOT가 필요한가?

단순 Hungarian + EKF의 한계:
- 클러터(허위 탐지)에 취약
- 새 물체 출현/소멸 처리가 휴리스틱
- 불확실한 연관에서 정보 손실

**확률론적 접근:** 연관 불확실성을 명시적으로 모델링.



```
IMM       : 여러 운동 모델 중 어느 모델인지 불확실할 때 → 혼합 추정
JPDA      : 탐지 ↔ 트랙 연관이 불확실할 때 → 확률 가중 업데이트
GM-PHD    : 트랙 수가 불확실할 때 → 강도 함수(intensity)로 표현
LMB       : 각 물체 존재 여부를 Bernoulli로 표현
PMBM      : PHD(새 물체) + 개별 Bernoulli(추적 중 물체) 통합
```




---

## 7-2. IMM (Interacting Multiple Model)

운동 모델 불확실성 처리: CV(등속), CA(등가속), CTRV(곡선) 혼합.



In [ ]:
import numpy as np

class IMMFilter:
    """
    models: [{'F': ..., 'H': ..., 'Q': ..., 'R': ...}, ...]
    mu    : 초기 모델 확률 [0.33, 0.33, 0.33]
    TPM   : 전이 확률 행렬 (n_models × n_models)
    """
    def __init__(self, models: list, mu: np.ndarray, TPM: np.ndarray):
        self.models = models
        self.mu     = mu.copy()
        self.TPM    = TPM
        n_x = models[0]['x0'].shape[0]
        self.xs = [m['x0'].copy() for m in models]
        self.Ps = [m['P0'].copy() for m in models]
        self.n_x = n_x

    def predict(self, dt: float):
        n = len(self.models)
        # 1. 혼합 확률 계산
        c_j = self.TPM.T @ self.mu           # (n,)
        mu_ij = (self.TPM * self.mu[:, None]) / c_j[None, :]  # (n, n)

        # 2. 혼합 초기 상태
        xs_mix = []
        Ps_mix = []
        for j in range(n):
            x_mix = sum(mu_ij[i, j] * self.xs[i] for i in range(n))
            P_mix = sum(mu_ij[i, j] * (self.Ps[i] +
                        np.outer(self.xs[i]-x_mix, self.xs[i]-x_mix))
                        for i in range(n))
            xs_mix.append(x_mix)
            Ps_mix.append(P_mix)

        # 3. 각 모델 독립 예측
        for j, m in enumerate(self.models):
            F = m['F'](dt) if callable(m['F']) else m['F']
            self.xs[j] = F @ xs_mix[j]
            self.P = F @ Ps_mix[j] @ F.T + m['Q']
            self.Ps[j] = self.P

        self.mu = c_j

    def update(self, z: np.ndarray):
        n = len(self.models)
        likelihoods = []
        for j, m in enumerate(self.models):
            H = m['H']
            y = z - H @ self.xs[j]
            S = H @ self.Ps[j] @ H.T + m['R']
            K = self.Ps[j] @ H.T @ np.linalg.inv(S)
            self.xs[j] = self.xs[j] + K @ y
            self.Ps[j] = (np.eye(self.n_x) - K @ H) @ self.Ps[j]
            # Likelihood
            pdf = np.exp(-0.5 * y.T @ np.linalg.inv(S) @ y)
            pdf /= np.sqrt((2*np.pi)**len(z) * np.linalg.det(S))
            likelihoods.append(float(pdf))

        # 4. 모델 확률 업데이트
        lam = np.array(likelihoods)
        self.mu = self.mu * lam
        self.mu /= self.mu.sum() + 1e-300

    @property
    def x(self):
        """혼합 추정값"""
        return sum(self.mu[j] * self.xs[j] for j in range(len(self.models)))

    @property
    def P(self):
        x_fused = self.x
        return sum(self.mu[j] * (self.Ps[j] +
                   np.outer(self.xs[j]-x_fused, self.xs[j]-x_fused))
                   for j in range(len(self.models)))




---

## 7-3. JPDA (Joint Probabilistic Data Association)

여러 트랙이 있을 때 연관 불확실성 → 확률 가중 업데이트.



In [ ]:
def jpda_update(tracks: list, measurements: list,
                PD: float = 0.9, PG: float = 0.99, lambda_c: float = 0.1):
    """
    PD       : 탐지 확률
    PG       : 게이팅 확률 (chi2 임계값 내)
    lambda_c : 클러터 밀도 (단위 체적당)

    출력: 각 트랙에 대한 beta (측정값별 연관 확률)
    """
    n_trk = len(tracks)
    n_meas = len(measurements)

    # 게이팅 + 이노베이션 행렬
    valid = np.zeros((n_trk, n_meas), dtype=bool)
    Lambdas = np.zeros((n_trk, n_meas))  # 혼합 가능도

    for i, trk in enumerate(tracks):
        for j, z in enumerate(measurements):
            y = z - trk['H'] @ trk['x']
            S = trk['H'] @ trk['P'] @ trk['H'].T + trk['R']
            # 마할라노비스 거리² < chi2(0.99, df)
            md2 = float(y.T @ np.linalg.inv(S) @ y)
            if md2 < 13.82:   # chi2(0.99, df=4) ≈ 13.28
                valid[i, j] = True
                det_S = np.linalg.det(S)
                L = np.exp(-0.5 * md2) / np.sqrt((2*np.pi)**len(z) * det_S)
                Lambdas[i, j] = L

    # 연관 확률 (단순 IPDA 근사)
    beta = np.zeros((n_trk, n_meas + 1))   # beta[i, 0] = 미연관
    for i in range(n_trk):
        denom = lambda_c * (1 - PD*PG)
        for j in range(n_meas):
            if valid[i, j]:
                denom += PD * Lambdas[i, j]
        for j in range(n_meas):
            if valid[i, j]:
                beta[i, j+1] = PD * Lambdas[i, j] / (denom + 1e-300)
        beta[i, 0] = 1.0 - beta[i, 1:].sum()

    # JPDA 업데이트 (확률 가중 혁신)
    updated_tracks = []
    for i, trk in enumerate(tracks):
        H = trk['H']
        x = trk['x'].copy()
        P = trk['P'].copy()

        y_fused = np.zeros_like(x[:H.shape[0]])
        for j, z in enumerate(measurements):
            if valid[i, j]:
                y_fused += beta[i, j+1] * (z - H @ x)

        K = P @ H.T @ np.linalg.inv(H @ P @ H.T + trk['R'])
        x_new = x + K @ y_fused

        # 확산 공분산 (spread of innovations)
        P_new = beta[i, 0] * P + (1 - beta[i, 0]) * (np.eye(len(x)) - K @ H) @ P
        for j, z in enumerate(measurements):
            if valid[i, j]:
                y_j = z - H @ x
                P_new += beta[i, j+1] * K @ (np.outer(y_j, y_j) - np.outer(y_fused, y_fused)) @ K.T

        updated_tracks.append({'x': x_new, 'P': P_new, **{k: v for k, v in trk.items() if k not in ('x', 'P')}})

    return updated_tracks, beta




---

## 7-4. GM-PHD (Gaussian Mixture Probability Hypothesis Density)

트랙 수를 추정하면서 복수 물체를 Gaussian 혼합으로 표현.



In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class GaussianComponent:
    weight: float
    mean:   np.ndarray
    cov:    np.ndarray

def gm_phd_predict(components: List[GaussianComponent],
                   F: np.ndarray, Q: np.ndarray,
                   p_survival: float = 0.99,
                   birth_components: List[GaussianComponent] = None):
    predicted = []
    # 생존 컴포넌트 예측
    for c in components:
        predicted.append(GaussianComponent(
            weight = p_survival * c.weight,
            mean   = F @ c.mean,
            cov    = F @ c.cov @ F.T + Q
        ))
    # 탄생 컴포넌트 추가
    if birth_components:
        predicted.extend(birth_components)
    return predicted

def gm_phd_update(predicted: List[GaussianComponent],
                  measurements: List[np.ndarray],
                  H: np.ndarray, R: np.ndarray,
                  p_detection: float = 0.9,
                  lambda_c: float = 1e-3):
    updated = []
    # 미탐지 컴포넌트
    for c in predicted:
        updated.append(GaussianComponent(
            weight = (1 - p_detection) * c.weight,
            mean   = c.mean.copy(),
            cov    = c.cov.copy()
        ))

    # 탐지된 측정값마다 업데이트 컴포넌트 생성
    for z in measurements:
        components_z = []
        for c in predicted:
            S = H @ c.cov @ H.T + R
            K = c.cov @ H.T @ np.linalg.inv(S)
            y = z - H @ c.mean
            L = (np.exp(-0.5 * y.T @ np.linalg.inv(S) @ y) /
                 np.sqrt((2*np.pi)**len(z) * np.linalg.det(S)))
            components_z.append(GaussianComponent(
                weight = p_detection * c.weight * L,
                mean   = c.mean + K @ y,
                cov    = (np.eye(len(c.mean)) - K @ H) @ c.cov
            ))
        # 정규화
        total = lambda_c + sum(gc.weight for gc in components_z)
        for gc in components_z:
            gc.weight /= total
        updated.extend(components_z)
    return updated

def gm_phd_prune(components: List[GaussianComponent],
                  threshold: float = 1e-5,
                  max_comp: int = 100,
                  merge_dist: float = 4.0) -> List[GaussianComponent]:
    # 가지치기
    comp = [c for c in components if c.weight > threshold]
    # 병합 (마할라노비스 거리 기준)
    merged = []
    used = [False] * len(comp)
    for i in range(len(comp)):
        if used[i]: continue
        group = [i]
        for j in range(i+1, len(comp)):
            if used[j]: continue
            d = comp[i].mean - comp[j].mean
            md2 = d.T @ np.linalg.inv(comp[i].cov) @ d
            if md2 < merge_dist:
                group.append(j)
                used[j] = True
        used[i] = True
        w_total = sum(comp[k].weight for k in group)
        x_merged = sum(comp[k].weight * comp[k].mean for k in group) / w_total
        P_merged = sum(comp[k].weight * (comp[k].cov +
                       np.outer(comp[k].mean - x_merged, comp[k].mean - x_merged))
                       for k in group) / w_total
        merged.append(GaussianComponent(w_total, x_merged, P_merged))
    # 상위 max_comp 유지
    merged.sort(key=lambda c: c.weight, reverse=True)
    return merged[:max_comp]

def extract_estimates(components: List[GaussianComponent],
                       threshold: float = 0.5) -> List[np.ndarray]:
    """가중치 > threshold 인 컴포넌트 → 상태 추정값"""
    estimates = []
    for c in components:
        n_objs = round(c.weight)
        for _ in range(n_objs):
            if c.weight > threshold:
                estimates.append(c.mean.copy())
    return estimates




---

## 7-5. PMBM (Poisson Multi-Bernoulli Mixture)

GM-PHD + LMB의 장점 결합. 새 물체는 PPP(Poisson Point Process), 추적 중 물체는 Bernoulli.



In [ ]:
@dataclass
class BernoulliComponent:
    r: float           # 존재 확률 (0~1)
    mean: np.ndarray   # 가우시안 평균
    cov:  np.ndarray   # 가우시안 공분산

class PMBM:
    """
    PPP  : 탐지되지 않은 새 물체 (클러터 + 탄생)
    MBM  : 개별 Bernoulli 컴포넌트 (추적 중 물체)
    """
    def __init__(self, pD=0.9, pS=0.99, lambda_c=1e-3):
        self.pD = pD
        self.pS = pS
        self.lambda_c = lambda_c
        self.ppp: List[GaussianComponent] = []    # PPP 강도 컴포넌트
        self.bernoullis: List[BernoulliComponent] = []

    def predict(self, F, Q, birth_ppp: List[GaussianComponent]):
        # PPP 예측
        self.ppp = [GaussianComponent(self.pS * c.weight, F @ c.mean, F @ c.cov @ F.T + Q)
                    for c in self.ppp] + birth_ppp
        # Bernoulli 예측
        for b in self.bernoullis:
            b.r    = self.pS * b.r
            b.mean = F @ b.mean
            b.cov  = F @ b.cov @ F.T + Q

    def update(self, measurements: List[np.ndarray], H, R):
        new_bernoullis = []

        for z in measurements:
            # 1. PPP 탐지 → 새 Bernoulli 생성 (data-driven birth)
            L_ppp_total = 0.0
            new_comp_candidates = []
            for c in self.ppp:
                S = H @ c.cov @ H.T + R
                K = c.cov @ H.T @ np.linalg.inv(S)
                y = z - H @ c.mean
                L = (self.pD * c.weight *
                     np.exp(-0.5 * y.T @ np.linalg.inv(S) @ y) /
                     np.sqrt((2*np.pi)**len(z) * np.linalg.det(S)))
                L_ppp_total += L
                new_comp_candidates.append((L, c.mean + K @ y, (np.eye(len(c.mean)) - K @ H) @ c.cov))

            delta_i = L_ppp_total
            denom = self.lambda_c + delta_i + sum(
                b.r * self.pD * self._likelihood(z, H @ b.mean, H @ b.cov @ H.T + R)
                for b in self.bernoullis
            )

            # 새 Bernoulli (PPP에서 탄생)
            if delta_i > 0:
                r_new = delta_i / denom
                x_new = sum(L * m for L, m, _ in new_comp_candidates) / delta_i
                P_new = sum(L * P for L, _, P in new_comp_candidates) / delta_i
                new_bernoullis.append(BernoulliComponent(r_new, x_new, P_new))

            # 기존 Bernoulli 업데이트
            for b in self.bernoullis:
                L_b = self._likelihood(z, H @ b.mean, H @ b.cov @ H.T + R)
                delta_b = b.r * self.pD * L_b
                r_upd = delta_b / denom
                S = H @ b.cov @ H.T + R
                K = b.cov @ H.T @ np.linalg.inv(S)
                y = z - H @ b.mean
                b.r    = r_upd
                b.mean = b.mean + K @ y
                b.cov  = (np.eye(len(b.mean)) - K @ H) @ b.cov

        # 미탐지 Bernoulli 업데이트
        for b in self.bernoullis:
            b.r = b.r * (1 - self.pD) / (1 - b.r * self.pD + 1e-300)

        self.bernoullis.extend(new_bernoullis)
        # 존재 확률 낮은 Bernoulli 제거
        self.bernoullis = [b for b in self.bernoullis if b.r > 0.1]

    def _likelihood(self, z, z_pred, S):
        y = z - z_pred
        return (np.exp(-0.5 * y.T @ np.linalg.inv(S) @ y) /
                np.sqrt((2*np.pi)**len(z) * np.linalg.det(S)))

    def extract_estimates(self, threshold=0.5):
        return [b.mean for b in self.bernoullis if b.r > threshold]




---

## 7-6. PointPainting + BEV 딥러닝 융합



In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

class PointPainter:
    """
    LiDAR 포인트에 카메라 세그멘테이션 점수 페인팅
    출력: (N, 3+C) 페인팅된 포인트 클라우드
    """
    CLASSES = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
    C = len(CLASSES)

    def __init__(self, model_path='yolov8m-seg.pt'):
        self.model = YOLO(model_path)

    def segment(self, img: np.ndarray) -> np.ndarray:
        """이미지 → (H, W, C) 클래스 점수 맵"""
        H, W = img.shape[:2]
        score_map = np.zeros((H, W, self.C), dtype=np.float32)
        results = self.model(img, verbose=False)[0]
        if results.masks is None:
            return score_map
        for mask, cls, conf in zip(results.masks.data,
                                    results.boxes.cls,
                                    results.boxes.conf):
            cls_idx = int(cls)
            cls_name = self.model.names[cls_idx]
            if cls_name not in self.CLASSES:
                continue
                score_map[:, :, self.CLASSES.index(cls_name)] = np.maximum(
                score_map[:, :, self.CLASSES.index(cls_name)],
                mask.cpu().numpy() * float(conf)
            )
        return score_map

    def paint(self, pts_lidar: np.ndarray, score_map: np.ndarray,
              R_L2C: np.ndarray, t_L2C: np.ndarray, K: np.ndarray) -> np.ndarray:
        """
        pts_lidar: (N, 3) LiDAR 포인트
        score_map: (H, W, C) 세그멘테이션 점수
        반환: (N, 3+C) 페인팅된 포인트
        """
        H, W, _ = score_map.shape
        # LiDAR → Camera 변환
        pts_cam = (R_L2C @ pts_lidar.T).T + t_L2C
        mask_front = pts_cam[:, 2] > 0
        scores = np.zeros((len(pts_lidar), self.C))

        # 카메라 FOV 내 포인트 투영
        uvw = (K @ pts_cam[mask_front].T).T
        uv = (uvw[:, :2] / uvw[:, 2:3]).astype(int)
        in_fov = ((uv[:, 0] >= 0) & (uv[:, 0] < W) &
                  (uv[:, 1] >= 0) & (uv[:, 1] < H))
        valid_uv = uv[in_fov]
        idx_map = np.where(mask_front)[0][in_fov]
        scores[idx_map] = score_map[valid_uv[:, 1], valid_uv[:, 0], :]

        return np.concatenate([pts_lidar, scores], axis=1)   # (N, 3+C)




---

## 7-7. 알고리즘 성능 비교 (nuScenes)

| 알고리즘 | MOTA | FP | FN | IDSW | 특징 |
|---------|------|----|----|------|------|
| EKF | -1.85 | 3241 | 892 | 18 | 기준선 |
| IMM | -1.45 | 2901 | 870 | 14 | 모델 다양성 |
| JPDA | -1.21 | 2580 | 841 | 9 | 연관 불확실성 |
| GM-PHD | -0.82 | 1890 | 823 | 6 | 클러터 강건 |
| LMB | -0.44 | 1120 | 801 | 5 | 존재 확률 추적 |
| **PMBM** | **-0.18** | **383** | **780** | **3** | **최고 성능** |

MOTA = 1 - (FP + FN + IDSW) / N_GT

---

## 7-8. 학습 로드맵 요약



```
기초                    중급                      고급
────────────────────────────────────────────────────────
LiDAR processing   →  Calibration         →  JPDA / GM-PHD
Camera features    →  KF / EKF / UKF      →  LMB / PMBM
FMCW radar        →  Multi-sensor fusion  →  PointPainting
                                          →  BEVFusion (딥러닝)
```




---

## 참고 자료

| 자료 | 내용 |
|------|------|
| Bar-Shalom et al. *Estimation with Applications to Tracking* | KF/EKF/IMM 이론 |
| Thrun et al. *Probabilistic Robotics* | Bayes 필터 기초 |
| Mahler *Statistical Multisource-Multitarget Information Fusion* | PHD/PMBM 이론 |
| Vora et al. *PointPainting* (CVPR 2020) | LiDAR-Camera sequential fusion |
| Liu et al. *BEVFusion* (CVPR 2023) | 딥러닝 BEV 융합 |
| nuScenes devkit | 멀티모달 데이터셋 평가 |
